# Part 4: Web source & MAI Grounding MCP

This notebook explores two paths for **web-grounded** knowledge:

1. The native `web` knowledge source kind (seen in inventory but **not fully
   validated** on the current BigBugs stamp).
2. **MAI Grounding** via `https://api.microsoft.ai/v3/mcp` — a real MCP server
   whose `web` tool works for direct probing, but cannot yet be consumed end-to-end
   via an `mcpServer` KB because the retrieve path is broken on this stamp.

### Prerequisites

| Item | Details |
|------|---------|
| `.env` file | In the `notebooks/` directory (same folder as this notebook) — `BIGBUGS_ENDPOINT`, `BIGBUGS_API_KEY`, `MAI_GROUNDING_ENDPOINT`, `MAI_GROUNDING_KEY`. |

> ⚠️ **The `.env` is currently in testing state — ask Matt for setup credentials.**
> Never check secrets into this repo.

### Current status on BigBugs

| Path | Status |
|------|--------|
| `web` KS kind | ⚠️ Seen in live inventory but **not probed** — shape/retrieve not validated |
| MAI Grounding direct MCP | ✅ `initialize`, `tools/list`, `tools/call` all work |
| `mcpServer` KS → KB → retrieve | ❌ Retrieve returns 500 (model-copy bug) or stub output |

## 1 — Setup

In [ ]:
import json, os
from pathlib import Path
from datetime import datetime, timezone

import requests

ENV_PATH = Path(".").resolve() / ".env"

def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        values[k.strip()] = v.strip()
    return values

env = load_env(ENV_PATH)
ENDPOINT = env["BIGBUGS_ENDPOINT"].rstrip("/")
API_KEY  = env["BIGBUGS_API_KEY"]
API_VERSION = "2026-05-01-preview"

MAI_ENDPOINT = env.get("MAI_GROUNDING_ENDPOINT", "https://api.microsoft.ai/v3/mcp").rstrip("/")
MAI_KEY      = env.get("MAI_GROUNDING_KEY", "")
MAI_KEY_HEADER = env.get("MAI_GROUNDING_KEY_HEADER", "x-apikey")

session = requests.Session()
session.headers.update({"api-key": API_KEY, "Accept": "application/json"})

def url(path: str) -> str:
    sep = "&" if "?" in path else "?"
    return f"{ENDPOINT}{path}{sep}api-version={API_VERSION}"

def pp(r: requests.Response):
    print(f"{r.status_code} {r.reason}")
    try:
        print(json.dumps(r.json(), indent=2))
    except Exception:
        print(r.text[:500])

stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")

print(f"Search Endpoint : {ENDPOINT}")
print(f"MAI Endpoint    : {MAI_ENDPOINT}")

## 2 — Probe the MAI Grounding MCP server directly

This validates that the upstream MCP server is alive and that the `web` tool works.

In [ ]:
mai_session = requests.Session()
mai_session.headers.update({
    MAI_KEY_HEADER: MAI_KEY,
    "Accept": "application/json, text/event-stream",
})

# Initialize
r = mai_session.post(MAI_ENDPOINT, json={
    "jsonrpc": "2.0", "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": "2024-11-05",
        "capabilities": {},
        "clientInfo": {"name": "lab532-notebook", "version": "1.0"},
    },
}, headers={"Content-Type": "application/json"})
print("=== initialize ===")
pp(r)

mcp_session_id = r.headers.get("mcp-session-id", "")
extra = {"mcp-session-id": mcp_session_id} if mcp_session_id else {}

In [ ]:
# tools/list
r = mai_session.post(MAI_ENDPOINT, json={
    "jsonrpc": "2.0", "id": 2,
    "method": "tools/list", "params": {},
}, headers={"Content-Type": "application/json", **extra})
print("=== tools/list ===")
pp(r)

In [ ]:
# tools/call — invoke the 'web' tool
r = mai_session.post(MAI_ENDPOINT, json={
    "jsonrpc": "2.0", "id": 3,
    "method": "tools/call",
    "params": {
        "name": "web",
        "arguments": {"query": "Azure AI Search knowledge base preview 2026"},
    },
}, headers={"Content-Type": "application/json", **extra}, timeout=60)
print("=== tools/call (web) ===")
pp(r)

## 3 — Stub: `web` knowledge source kind

The `web` kind exists in the live service inventory but has **not been validated**
in the investigation. The shape below is speculative — exercise with caution.

> ❌ **This section is a stub.** The exact KS create shape for `web` is not
> confirmed and may fail.

In [ ]:
# Speculative web KS shape — may need adjustment once the
# product team confirms the contract.
WEB_KS_NAME = f"lab532-web-{stamp}"

web_ks_body = {
    "name": WEB_KS_NAME,
    "kind": "web",
    "description": "Lab 532 web source (speculative shape)",
}

r = session.put(url(f"/knowledgesources('{WEB_KS_NAME}')"), json=web_ks_body,
                headers={"Prefer": "return=representation"})
pp(r)

if r.status_code in (200, 201):
    print("\n✅ Web KS created — try wiring it into a KB and running retrieve.")
    # Cleanup
    session.delete(url(f"/knowledgesources('{WEB_KS_NAME}')"))
else:
    print("\n⚠️ Web KS create failed — the exact shape is not confirmed on this stamp.")

## 4 — Stub: MAI Grounding as an `mcpServer` KS

The MAI Grounding MCP endpoint works directly (see above), but wiring it through
an `mcpServer` knowledge source is **blocked** on the current stamp:

- KB create requires `retrievalReasoningEffort = low` and a valid chat model.
- KB retrieve with `knowledgeSourceParams.kind = "mcpServer"` returns **500**
  (model-copy deserialization bug).
- Without `knowledgeSourceParams`, retrieve returns only stub content
  (`"Stub MCP server result"`).

This section shows the **setup** so you can re-test when the stamp is fixed.

In [ ]:
MAI_KS_NAME = f"lab532-mai-mcp-{stamp}"
MAI_KB_NAME = f"lab532-mai-mcp-kb-{stamp}"
AZURE_OPENAI_ENDPOINT = env.get("AZURE_OPENAI_ENDPOINT", "").rstrip("/") + "/"
AZURE_OPENAI_KEY = env.get("AZURE_OPENAI_KEY", "")

# Create KS
mai_ks_body = {
    "name": MAI_KS_NAME,
    "kind": "mcpServer",
    "description": "Lab 532 MAI Grounding MCP",
    "mcpServerParameters": {
        "serverURL": MAI_ENDPOINT,
        "authentication": {
            "kind": "storedHeaders",
            "storedHeadersParameters": {
                "headers": {MAI_KEY_HEADER: MAI_KEY}
            },
        },
        "tools": [
            {"name": "web", "outputParsing": {"kind": "auto"}}
        ],
    },
}

r = session.put(url(f"/knowledgesources('{MAI_KS_NAME}')"), json=mai_ks_body,
                headers={"Prefer": "return=representation"})
print("=== Create MAI MCP KS ===")
pp(r)

In [ ]:
# Create KB (needs low reasoning + chat model)
mai_kb_body = {
    "name": MAI_KB_NAME,
    "description": "Lab 532 KB — MAI Grounding MCP (stub)",
    "outputMode": "extractiveData",
    "retrievalReasoningEffort": {"kind": "low"},
    "knowledgeSources": [{"name": MAI_KS_NAME}],
    "models": [
        {
            "kind": "azureOpenAI",
            "azureOpenAIParameters": {
                "resourceUri": AZURE_OPENAI_ENDPOINT,
                "deploymentId": "gpt-5.4-mini",
                "modelName": "gpt-5.4-mini",
                "apiKey": AZURE_OPENAI_KEY,
            },
        }
    ],
}

r = session.put(url(f"/knowledgebases('{MAI_KB_NAME}')"), json=mai_kb_body,
                headers={"Prefer": "return=representation"})
print("=== Create MAI MCP KB ===")
pp(r)

In [ ]:
# Retrieve — expected to FAIL on current stamp
retrieve_body = {
    "includeActivity": True,
    "messages": [
        {"role": "user", "content": [{"type": "text", "text": "What is new in Azure AI Search 2026 preview?"}]}
    ],
    "knowledgeSourceParams": [
        {
            "kind": "mcpServer",
            "knowledgeSourceName": MAI_KS_NAME,
            "includeReferenceSourceData": True,
            "includeReferences": True,
            "alwaysQuerySource": True,
        }
    ],
    "maxRuntimeInSeconds": 120,
}

r = session.post(url(f"/knowledgebases('{MAI_KB_NAME}')/retrieve"), json=retrieve_body, timeout=180)
print("=== Retrieve (expected to fail) ===")
pp(r)

if r.status_code == 500:
    print("\n❌ Expected: mcpServer retrieve hits the model-copy deserialization bug.")
    print("   This will be fixed in a future stamp update.")
elif r.status_code == 200:
    body = r.json()
    resp_items = body.get("response", [])
    for item in resp_items:
        for c in item.get("content", []):
            text = c.get("text", "")
            if "Stub" in text:
                print("\n⚠️ Got 200 but content is stub — real MCP retrieval not wired yet.")
            else:
                print("\n✅ Got real MCP-grounded content!")

In [ ]:
# Cleanup
for label, path in [
    ("MAI KB", f"/knowledgebases('{MAI_KB_NAME}')"),
    ("MAI KS", f"/knowledgesources('{MAI_KS_NAME}')"),
]:
    r = session.delete(url(path))
    print(f"Delete {label}: {r.status_code} {r.reason}")

## Summary

| Approach | Status |
|----------|--------|
| Direct MAI Grounding MCP probe | ✅ Works — `initialize`, `tools/list`, `tools/call` all OK |
| `web` KS kind via KB | ⚠️ Shape not validated on this stamp |
| `mcpServer` KS (MAI Grounding) via KB retrieve | ❌ Blocked — 500 on retrieve, stub fallback |

➡️ Continue to [Part 5: Add arbitrary MCP servers](part5-arbitrary-mcp-servers.ipynb).